# 00 - Setup

This notebook prepares the local workshop environment:
- Create and configure `.env` with your API key
- Install/sync dependencies with `uv`
- Start Qdrant in Docker and verify it is reachable


In [2]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'docker-compose.yml').exists():
            return candidate
    raise RuntimeError('Could not find repository root (docker-compose.yml missing).')

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
ENV_PATH = NOTEBOOKS_DIR / '.env'
print('Repo root:', REPO_ROOT)
print('Notebooks dir:', NOTEBOOKS_DIR)
print('.env path:', ENV_PATH)


Repo root: /home/hanmul/Workshops/workshop-ragV2
Notebooks dir: /home/hanmul/Workshops/workshop-ragV2/notebooks
.env path: /home/hanmul/Workshops/workshop-ragV2/notebooks/.env


## 1) Configure `.env` (API key)

Set your API key in `notebooks/.env`.

Expected key for notebook experiments:
- `OPENAI_API_KEY`

> Do not commit real secrets to git.


In [3]:
from pathlib import Path

template = '''# Notebook environment
OPENAI_API_KEY=your_openai_api_key_here

# Optional Qdrant override (defaults used in this notebook)
QDRANT_HOST=localhost
QDRANT_PORT=6333
'''.strip() + '\n'

if not ENV_PATH.exists():
    ENV_PATH.write_text(template, encoding='utf-8')
    print(f'Created {ENV_PATH}')
else:
    print(f'{ENV_PATH} already exists')

print('Open the file and replace OPENAI_API_KEY with your real key before running later notebooks.')


Created /home/hanmul/Workshops/workshop-ragV2/notebooks/.env
Open the file and replace OPENAI_API_KEY with your real key before running later notebooks.


In [4]:
import os

# lightweight .env reader (no extra dependency required)
for line in ENV_PATH.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#') or '=' not in line:
        continue
    key, value = line.split('=', 1)
    os.environ[key.strip()] = value.strip()

api_key = os.getenv('OPENAI_API_KEY', '')
if not api_key or api_key == 'your_openai_api_key_here':
    raise ValueError('OPENAI_API_KEY is not configured in notebooks/.env yet.')

print('OPENAI_API_KEY is configured (value hidden).')


OPENAI_API_KEY is configured (value hidden).


## 2) Start Qdrant in Docker

This starts (or reuses) a local `qdrant` container with persistent storage in `qdrant_storage/`.


In [5]:
import subprocess

storage_dir = REPO_ROOT / 'qdrant_storage'
storage_dir.mkdir(exist_ok=True)

docker_check = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if docker_check.returncode != 0:
    raise RuntimeError('Docker is required to start Qdrant from this notebook.')

exists = subprocess.run(['docker', 'ps', '-aq', '-f', 'name=^qdrant$'], capture_output=True, text=True)
container_id = exists.stdout.strip()

if container_id:
    running = subprocess.run(['docker', 'ps', '-q', '-f', 'name=^qdrant$'], capture_output=True, text=True)
    if running.stdout.strip():
        print('Qdrant container is already running.')
    else:
        subprocess.run(['docker', 'start', 'qdrant'], check=True)
        print('Started existing qdrant container.')
else:
    subprocess.run([
        'docker', 'run', '-d',
        '--name', 'qdrant',
        '--restart', 'unless-stopped',
        '-p', '6333:6333',
        '-v', f'{storage_dir}:/qdrant/storage',
        'qdrant/qdrant:latest',
    ], check=True)
    print('Started new qdrant container.')

print('Qdrant dashboard: http://localhost:6333/dashboard')


Qdrant container is already running.
Qdrant dashboard: http://localhost:6333/dashboard


In [6]:
import json
from urllib.request import urlopen

with urlopen('http://localhost:6333') as resp:
    payload = json.loads(resp.read().decode('utf-8'))

print('Qdrant is reachable:')
print(payload)


Qdrant is reachable:
{'title': 'qdrant - vector search engine', 'version': '1.16.3', 'commit': 'bd49f45a8a2d4e4774cac50fa29507c4e8375af2'}


## Next
Run the next notebook after this setup is complete:
- `01_chunking_and_retrieval.ipynb`
